# Data Science for Business - Fair ML for Credit Scoring

*Session 13 · Beyond ISLP; uses 4.3 (logistic regression) and 5.1.1 (validation set) · External: Barocas, Hardt & Narayanan, Fairness and Machine Learning (fairmlbook.org), Ch. 3; Kamiran & Calders (2012, reweighing); Kamiran et al. (2012, reject option); AIF360 docs*

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
# Install packages that are not pre-installed on Colab
# !pip install aif360

In [ ]:
import logging
import warnings

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# aif360 logs a warning for every optional dependency that is not installed (fairlearn, inFairness, ...).
# We only need its core functions, so we raise the logging level to hide these messages.
logging.getLogger().setLevel(logging.ERROR)
# Some aif360 source files contain invalid escape sequences, which Python 3.12 reports once when compiling them.
warnings.filterwarnings('ignore', message='invalid escape sequence', category=SyntaxWarning)

from aif360.sklearn.datasets import fetch_german
from aif360.sklearn.metrics import disparate_impact_ratio, equal_opportunity_difference
from aif360.sklearn.preprocessing import Reweighing
from aif360.sklearn.postprocessing import RejectOptionClassifier

In [ ]:
np.random.seed(42)

## Problem description

In this notebook we will explore fairness in machine learning models for credit scoring using the German Credit dataset (https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data). The goal is to predict whether an individual has good or bad credit risk while ensuring that the model does not discriminate based on sensitive attributes such as gender or age.

Throughout the notebook, the **protected attribute** is `sex` (1 = male = privileged group, 0 = female = unprivileged group), and the **favourable outcome** is good credit (1).

Fairness is not covered in ISLP. Further reading: S. Barocas, M. Hardt & A. Narayanan, *Fairness and Machine Learning* (<https://fairmlbook.org>), Ch. 3 (classification), and the AIF360 documentation (<https://aif360.readthedocs.io>).

## Load data

First, we load the German Credit data from the AIF360 package. For more information, see: https://aif360.readthedocs.io/en/stable/modules/generated/aif360.sklearn.datasets.fetch_german.html

We use the numeric version of the data (9 predictors). The protected attributes `sex`, `age` (binarized: older than 25) and `foreign_worker` are stored in the index of `X` and `y`, which is where the AIF360 functions look for them.

In [ ]:
data = fetch_german(numeric_only=True)

In [ ]:
data.X

In [ ]:
data.y

## Prepare data

We split the data into a training set (80%) and a test set (20%) (validation set approach, ISLP 5.1.1). As only 30% of the applicants have bad credit, we stratify the split by the response so that both sets have the same share of good and bad credit.

In [ ]:
X = data.X
y = data.y

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

The predictors are on very different scales (e.g., `credit_amount` in DM vs. `num_dependents`), so we standardize them. Instead of scaling the data by hand, we will put the scaler and the model into one scikit-learn `Pipeline` (next section). A pipeline is a chain of steps that behaves like a single model: `fit` fits all steps in order on the training data, and `predict` applies the same (already fitted) steps to new data. This guarantees that the scaler only learns from the training data (no leakage from the test set, and none from held-out folds if we ever cross-validate), and we only have one object to fit and predict with.

`set_output(transform='pandas')` makes the scaler return a dataframe with the original index, so AIF360 can still find the protected attributes.

In [ ]:
scaler = StandardScaler().set_output(transform='pandas')
scaler

## Train classifier

Next, we train a standard logistic regression (ISLP 4.3) without any fairness constraints.

Note that `sex` (as well as `age` and `foreign_worker`) is one of the input features here. Would dropping it ("fairness through unawareness") solve the problem? Usually not: other features (e.g., credit amount or duration) can act as **proxies** for the protected attribute. Keep this question in mind; you will test it at the end of the notebook.

`LogisticRegression` in scikit-learn is L2-penalised by default (`C=1`, the inverse of the penalty λ). A huge `C` (here `1e10`) makes the penalty negligible, which gives the plain, unpenalised logistic regression of ISLP 4.3.

We chain the scaler and the logistic regression into a pipeline and fit it on the (unscaled) training data. The steps get names (`'scaler'`, `'model'`) so that we can access them later.

In [ ]:
pipe_logit = Pipeline([
    ('scaler', scaler),
    ('model', LogisticRegression(C=1e10, max_iter=1000))  # C = 1/λ; huge C ≈ no penalty
])
pipe_logit.fit(X_train, y_train)

The estimated coefficients (on standardized predictors) show how each feature, including `sex`, enters the model. We get the fitted model and the feature names from the pipeline's named steps.

In [ ]:
pd.Series(pipe_logit.named_steps['model'].coef_[0], index=pipe_logit.named_steps['scaler'].get_feature_names_out())

Predict the classes (threshold 0.5) and the probabilities for the test set. The pipeline scales the test data with the scaler fitted on the training data.

In [ ]:
y_pred = pipe_logit.predict(X_test)
y_proba = pipe_logit.predict_proba(X_test)

We report two performance metrics: accuracy and the F1 score for the favourable class (good credit).

In [ ]:
print('Accuracy:', accuracy_score(y_test, y_pred))
print('F1 score:', f1_score(y_test, y_pred))

## Compute fairness metrics

Let $A$ be the protected attribute ($A=1$: privileged, $A=0$: unprivileged) and $\hat Y$ the predicted label ($\hat Y=1$: good credit). We use three common group fairness metrics:

- **Statistical parity difference (SPD)** $= P(\hat Y=1 \mid A=0) - P(\hat Y=1 \mid A=1)$: difference in approval rates. Fair = 0.
- **Disparate impact ratio (DI)** $= P(\hat Y=1 \mid A=0) \,/\, P(\hat Y=1 \mid A=1)$: the same comparison as a ratio. Fair = 1; the "four-fifths rule" flags values below 0.8 (we also show 1.25 as the symmetric upper bound).
- **Equal opportunity difference (EOD)** $= TPR_{A=0} - TPR_{A=1}$, where $TPR = P(\hat Y=1 \mid Y=1, A)$: among creditworthy people, are both groups approved equally often? Fair = 0.

Negative EOD/SPD and DI below 1 mean that the privileged group benefits; positive EOD/SPD and DI above 1 mean that the unprivileged group benefits.

A word of caution: the test set contains only 200 applicants, of whom only about 60 are women. A single changed decision can move these metrics by a few percentage points, so small differences should not be over-interpreted.

### Equal opportunity difference

Here we compute the equal opportunity difference before applying any fairness mitigation techniques.

In [ ]:
eod = equal_opportunity_difference(y_test, y_pred, prot_attr='sex', priv_group=1)
print(eod)

sns.barplot(x=['Equal Opportunity Difference'], y=[eod])
plt.axhline(y=0.0, color='green', linestyle='--')
plt.fill_between(x=[-0.5, 0.5], y1=-0.1, y2=0.1, color='green', alpha=0.2)
plt.title('Equal Opportunity Difference for Sex')
plt.ylim(-1.0, 1.0)
plt.show()

A negative value implies a higher benefit for the privileged group. However, the value is close to zero, indicating only a small disparity between groups.

### Disparate impact ratio

Similarly, we compute the disparate impact ratio.

In [ ]:
di = disparate_impact_ratio(y_test, y_pred, prot_attr='sex', priv_group=1)
print(di)

sns.barplot(x=['Disparate Impact Ratio'], y=[di])
plt.axhline(y=1.0, color='green', linestyle='--')
plt.fill_between(x=[-0.5, 0.5], y1=0.8, y2=1.25, color='green', alpha=0.2)
plt.title('Disparate Impact Ratio for Sex')
plt.ylim(0, 2)
plt.show()

Again, a value less than 1 implies a higher benefit for the privileged group. However, the value is within the bounds of the "four-fifths rule", suggesting that the disparity is not substantial.

Note that DI ignores the true labels, while EOD conditions on them. If the base rates of good credit differ between the groups, both criteria generally cannot be satisfied at the same time. Choosing a fairness metric is therefore a normative decision, not a purely technical one (see fairmlbook.org, Ch. 3).

## Bias mitigation algorithms

Although the fairness metrics indicate only a small disparity, we will still apply two common bias mitigation techniques to see their effect on fairness and predictive performance.

### Pre-processing algorithms: Reweighing

Reweighing (Kamiran & Calders, 2012) assigns a weight to each (group, label) cell of the training data: $w = P(A)\,P(Y) \,/\, P(A, Y)$. In the weighted data, sex and the label are independent. The features themselves are unchanged, so we can compute the weights on the unscaled training data.

In [ ]:
rw_preprocessor = Reweighing(prot_attr='sex')
X_train_rw, new_weights = rw_preprocessor.fit_transform(X_train, y_train)

Display the weights assigned to each training observation. Combinations that are rarer than independence would imply (e.g., women with good credit) get weights above 1.

In [ ]:
new_weights

Retrain the classifier using the computed sample weights. A pipeline passes extra fit arguments to a step if we prefix them with the step name and two underscores: `model__sample_weight` goes to the `'model'` step (the scaler itself is not weighted).

In [ ]:
pipe_logit_rw = Pipeline([
    ('scaler', StandardScaler().set_output(transform='pandas')),
    ('model', LogisticRegression(C=1e10, max_iter=1000))  # same model as above, now with sample weights
])
pipe_logit_rw.fit(X_train, y_train, model__sample_weight=new_weights)

In [ ]:
y_pred_rw = pipe_logit_rw.predict(X_test)

Compute the equal opportunity difference and disparate impact ratio after applying reweighing.

In [ ]:
eod_rw = equal_opportunity_difference(y_test, y_pred_rw, prot_attr='sex', priv_group=1)
print(eod_rw)

sns.barplot(x=['Equal Opportunity Difference'], y=[eod_rw])
plt.axhline(y=0.0, color='green', linestyle='--')
plt.fill_between(x=[-0.5, 0.5], y1=-0.1, y2=0.1, color='green', alpha=0.2)
plt.title('Equal Opportunity Difference for Sex')
plt.ylim(-1.0, 1.0)
plt.show()

In [ ]:
di_rw = disparate_impact_ratio(y_test, y_pred_rw, prot_attr='sex', priv_group=1)
print(di_rw)

sns.barplot(x=['Disparate Impact Ratio'], y=[di_rw])
plt.axhline(y=1.0, color='green', linestyle='--')
plt.fill_between(x=[-0.5, 0.5], y1=0.8, y2=1.25, color='green', alpha=0.2)
plt.title('Disparate Impact Ratio for Sex')
plt.ylim(0, 2)
plt.show()

Both metrics have moved to the other side of the fairness line: now the unprivileged group benefits slightly (EOD larger than 0 and DI larger than 1). Given the small test set, however, both the before and after values are within a few changed decisions of perfect parity.

What did reweighing cost in terms of predictive performance?

In [ ]:
print('Accuracy:', accuracy_score(y_test, y_pred_rw))
print('F1 score:', f1_score(y_test, y_pred_rw))

Accuracy and F1 score are almost unchanged compared to the original classifier. On 200 test observations, differences of this size are within noise.

### Post-processing algorithms: Reject option classification

Reject option classification (ROC; Kamiran et al., 2012) does not change the model, only its decisions. Inside an uncertain band around the decision threshold 0.5, ROC gives the favourable label to the unprivileged group and the unfavourable label to the privileged group. Outside the band, the original predictions are kept. A wider band means more fairness adjustment, but also more changed decisions.

Here, we choose a margin of 0.02 around the threshold (i.e., the band [0.48, 0.52]).

In [ ]:
roc_postprocessor = RejectOptionClassifier(prot_attr='sex', threshold=0.5, margin=0.02)

As ROC is a post-processing technique, we do not need to retrain the classifier. ROC works on the **predicted probabilities** of the original classifier, so we fit it on the training scores and then apply it to the test scores. We wrap the scores in a dataframe with the original index, so ROC knows each applicant's sex.

In [ ]:
y_proba_train = pipe_logit.predict_proba(X_train)
roc_postprocessor.fit(pd.DataFrame(y_proba_train, index=y_train.index), y_train)

In [ ]:
y_pred_roc = roc_postprocessor.predict(pd.DataFrame(y_proba, index=y_test.index))

How many test decisions did ROC change?

In [ ]:
(y_pred_roc != y_pred).sum()

Check the fairness metrics after reject option classification.

In [ ]:
eod_roc = equal_opportunity_difference(y_test, y_pred_roc, prot_attr='sex', priv_group=1)
print(eod_roc)

sns.barplot(x=['Equal Opportunity Difference'], y=[eod_roc])
plt.axhline(y=0.0, color='green', linestyle='--')
plt.fill_between(x=[-0.5, 0.5], y1=-0.1, y2=0.1, color='green', alpha=0.2)
plt.title('Equal Opportunity Difference for Sex')
plt.ylim(-1.0, 1.0)
plt.show()

In [ ]:
di_roc = disparate_impact_ratio(y_test, y_pred_roc, prot_attr='sex', priv_group=1)
print(di_roc)

sns.barplot(x=['Disparate Impact Ratio'], y=[di_roc])
plt.axhline(y=1.0, color='green', linestyle='--')
plt.fill_between(x=[-0.5, 0.5], y1=0.8, y2=1.25, color='green', alpha=0.2)
plt.title('Disparate Impact Ratio for Sex')
plt.ylim(0, 2)
plt.show()

With this narrow band, ROC changes only a handful of decisions, and both metrics move closer to parity (EOD closer to 0 and DI closer to 1). Compared to reweighing, the values look slightly "fairer", but with only about 60 women in the test set, this difference amounts to one or two applicants and should not be read as evidence that one method is better than the other.

And the predictive performance:

In [ ]:
print('Accuracy:', accuracy_score(y_test, y_pred_roc))
print('F1 score:', f1_score(y_test, y_pred_roc))

Again, accuracy and F1 score barely change compared to the original classifier. In this example, both mitigation techniques adjust the fairness metrics at almost no cost in predictive performance, but all differences are small and noisy. A proper comparison would use cross-validation or repeated splits.

## Your turn!

**Your turn!** Drop `sex` from the predictors (fairness through unawareness), retrain the logistic regression and recompute the equal opportunity difference and the disparate impact ratio. Do the fairness metrics change? Why (not)? (Hint: the protected attribute stays in the index, so the AIF360 metrics still work.)

In [ ]:
# YOUR CODE HERE

**Your turn!** Repeat the reject option classification with wider margins (e.g., 0.05 and 0.1). How many decisions change, and how do the fairness metrics and accuracy react?

In [ ]:
# YOUR CODE HERE